# 05 ? Frozen v1_crop128: collect evidence and review three conditions

CPU audit only. No preprocessing, relabeling, splitting, checkpoint selection or training.
Decision remains `KEEP_V1_WITH_CONDITIONS` until complete evidence and human review.
Run all cells in order on a fresh Colab runtime. See `docs/dataset_audit_review.md`.


In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
BRANCH = "fix/dataset-audit-mobilenetv2"
REPO = Path('/content/audit_review_repo')
REPO_URL = 'https://github.com/phanyen06ym-ien/ocean-plastic-ml-starter.git'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--single-branch', '-b', BRANCH, REPO_URL, str(REPO)], check=True)
else:
    dirty = subprocess.run(['git', '-C', str(REPO), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout.strip()
    if dirty:
        raise RuntimeError('Use a clean runtime or preserve local changes before updating this checkout.')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', 'FETCH_HEAD'], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
COMMIT_SHA = subprocess.run(['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
print('Branch:', BRANCH, 'SHA:', COMMIT_SHA)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-mobilenet.txt'], check=True)


## Restore frozen data; configure explicit source metadata

The ZIP must contain `v1_crop128/`. Set both COCO paths if the original Instance metadata is available. Defaults deliberately leave metadata missing and produce `UNVERIFIED`; no video IDs will be invented. These files must be the exports used for the frozen manifest, including their complete `images`, `annotations`, `categories` lists. Do not recreate v1 if the ZIP is unavailable.


In [ ]:
DRIVE = Path('/content/drive/MyDrive/ocean-plastic-ml')
PROCESSED = Path('/content/processed/v1_crop128')
PROCESSED_ZIP = DRIVE / 'data/processed/v1_crop128.zip'
if not PROCESSED.exists():
    if not PROCESSED_ZIP.is_file():
        raise FileNotFoundError(f'Restore the frozen ZIP on Drive: {PROCESSED_ZIP}')
    PROCESSED.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', str(PROCESSED_ZIP), '-d', str(PROCESSED.parent)], check=True)
for name in ['split_manifest.csv', 'class_mapping.json', 'preprocessing_summary.json']:
    assert (PROCESSED / name).is_file(), f'Missing frozen input: {name}'

# Fill these with exact original COCO files; None is supported, not treated as PASS.
SOURCE_TRAIN_COCO = None
SOURCE_VAL_COCO = None
SOURCES = {s: p for s, p in [('train', SOURCE_TRAIN_COCO), ('val', SOURCE_VAL_COCO)] if p}
HASH_SIZE = 8
THRESHOLD = 6  # inclusive Hamming distance out of 64 bits; screening only
MAX_PAIR_IMAGES = 50  # limits PNG previews, never limits candidate CSV or scan coverage
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = DRIVE / 'runs/data_audit_v1_crop128/review' / RUN_ID
print('Output:', OUT)


In [ ]:
# Full regression suite: no dataset mutation or model training.
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


In [ ]:
from src.audit_review import collect_review
report = collect_review(PROCESSED, OUT, source_cocos=SOURCES,
                        threshold=THRESHOLD, hash_size=HASH_SIZE,
                        max_pair_images=MAX_PAIR_IMAGES)
print(report['status'], report['decision'])
print(json.dumps({k: v['status'] for k, v in report['conditions'].items()}, indent=2))
print('Missing video evidence:', report['video_reconciliation']['missing_evidence'])
print('Video conflicts:', report['video_reconciliation']['conflicts'])
print('Split-pair scan:', report['near_duplicate_scan']['split_pairs'])
print('Summary:', OUT / 'audit_review_summary.json')
# Check evidence production, not a human PASS or proof that leakage is absent.
assert report['automated_checks_passed'], 'Inspect audit_summary.json; integrity errors block continuation'
assert all(c['status'] == 'UNVERIFIED' for c in report['conditions'].values())
assert report['decision'] == 'KEEP_V1_WITH_CONDITIONS'
assert report['near_duplicate_scan']['complete_cross_split_scan']
assert (OUT / 'evidence_index.json').is_file()


In [ ]:
# Show a label contact sheet; every tile maps to a row in rare_class_review.csv.
from IPython.display import display, Image as DisplayImage
pages = report['rare_class_scope']['contact_sheets']
if pages:
    display(DisplayImage(filename=str(OUT / pages[0])))
print('Label rows:', report['rare_class_scope']['review_rows'])
print('Pair preview count / omitted:', report['near_duplicate_scan']['example_count'],
      report['near_duplicate_scan']['examples_omitted'])


## Human review (after collection)

Copy `rare_class_review.csv` and `near_duplicate_candidates.csv` to new `*_reviewed.csv` files. Preserve every row and metadata field. Fill `decision` (`PASS`, `FAIL`, `UNCERTAIN`), `reviewer`, `evidence`, `notes` on every row. PASS for a pair means context review did not substantiate leakage; inspect originals/video context and acknowledge hash limitations. Label review requires original image and annotation, not just the contact sheet.

Copy `condition_decisions.template.json` to `condition_decisions.json`, preserving `run_id`; fill all four fields for each condition only when supported. The video condition also requires review of `video_reconciliation.json/.csv` and source provenance. Do not edit immutable evidence. A zero-candidate scan still requires human assessment of method/threshold/limitations.

The next cell is safe before reviewers finish: absent decisions remain UNVERIFIED. Re-run it after completing review. If source metadata was missing, collect a fresh run with both source paths; do not retrofit the old evidence.


In [ ]:
from src.audit_review import summarize_review, write_json
DECISIONS = OUT / 'condition_decisions.json'
PAIR_REVIEW = OUT / 'near_duplicate_candidates_reviewed.csv'
LABEL_REVIEW = OUT / 'rare_class_review_reviewed.csv'
reviewed = summarize_review(OUT,
    decisions_path=DECISIONS if DECISIONS.is_file() else None,
    pair_review=PAIR_REVIEW if PAIR_REVIEW.is_file() else None,
    label_review=LABEL_REVIEW if LABEL_REVIEW.is_file() else None)
REVIEWED_OUTPUT = OUT / ('audit_review_summary_reviewed_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json')
write_json(REVIEWED_OUTPUT, reviewed)
print(reviewed['status'], reviewed['decision'])
print(json.dumps({k: v['status'] for k, v in reviewed['conditions'].items()}, indent=2))
print('Reviewed output:', REVIEWED_OUTPUT)
